# Face-aging inference

Load one trained `.pt` checkpoint, provide one source photograph and either a numeric age or an explicit aging prompt, then save the aged result. The same API supports direct img2img editing and deterministic DDIM inversion. Run this notebook in the existing `deep_learning` Conda environment on the server.

In [ ]:
from pathlib import Path
import torch
from PIL import Image
from IPython.display import display

from src.inference import (
    compare_inference_modes,
    diagnose_checkpoint_age_sweep,
    generate_age_sweep,
    infer_face_aging,
    load_face_aging_inference_bundle,
    run_inference_pipeline_validation,
    save_inference_image,
)

## 1. Paths and editing request

`CHECKPOINT_PATH` accepts either `adapter_inference.pt` or `training_resume.pt`. Set `TARGET_AGE` and leave `TARGET_PROMPT=None` for the training-compatible automatic prompt, or provide an explicit prompt. If both are supplied, the explicit prompt wins and an age mismatch produces a warning.

In [ ]:
CHECKPOINT_PATH = Path('/server/checkpoints/face_aging/best/adapter_inference.pt')
SOURCE_IMAGE = Path('/server/images/person.jpg')
OUTPUT_DIR = Path('/server/results/face_aging')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_AGE = 65
TARGET_PROMPT = None  # e.g. 'photo of a person as 65-year-old'
SOURCE_AGE = None     # useful for inversion; generic prompt is used if unknown
SOURCE_PROMPT = None

# Main switch requested for inverse diffusion.
USE_INVERSE_DIFFUSION = True
SEED = 42

## 2. Reconstruct SD1.5 and load trained weights

The checkpoint stores LoRA, expanded `conv_in`, base model ID, adapter configuration and VAE metadata—not a duplicate frozen SD1.5 U-Net. The loader reconstructs that base model and then loads only the trained tensors. No optimizer, GradScaler or DataLoader is required.

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    MODEL_DTYPE = torch.float32

bundle = load_face_aging_inference_bundle(
    CHECKPOINT_PATH,
    device=DEVICE,
    dtype=MODEL_DTYPE,
    local_files_only=False,
    load_auxiliary_models=True,
    auxiliary_dtype=torch.float32,
    auxiliary_trust_remote_code=True,  # review/pin the MiVOLO revision in production
)
bundle['inference_checkpoint_report']

## 3. Generate and save one aged image

Direct mode starts from the encoded source and adds noise according to `strength`; lower values preserve more structure. Inverse mode first performs deterministic DDIM inversion under the source prompt, then denoises under the target prompt. `use_inverse_diffusion` overrides `mode`, making the experiment flag explicit. Three-way CFG uses full, image-only and unconditional branches in one U-Net call.

In [ ]:
result = infer_face_aging(
    bundle=bundle,
    image=SOURCE_IMAGE,
    target_age=TARGET_AGE,
    target_prompt=TARGET_PROMPT,
    source_age=SOURCE_AGE,
    source_prompt=SOURCE_PROMPT,

    use_inverse_diffusion=USE_INVERSE_DIFFUSION,
    num_inference_steps=50,
    strength=0.45,            # used by direct mode
    inversion_strength=1.0,  # used by inverse mode
    text_guidance_scale=7.0,
    image_guidance_scale=1.5,
    negative_prompt='',
    seed=SEED,
    image_size=256,
    output_type='pil',
)

mode_name = result['mode']
saved_path = save_inference_image(
    result, OUTPUT_DIR / f'aged_{TARGET_AGE}_{mode_name}_seed{SEED}.png'
)
print('Saved:', saved_path)
print('Diagnostics:', result['diagnostics'])
display(result['image'])
result['metadata']

## 6. Diagnose one saved checkpoint

This reloads the selected `.pt`, generates every requested age with one fixed seed, prints age calibration and identity preservation, and saves the individual images, annotated grid, and CSV.

In [ ]:
DIAGNOSTIC_DIR = OUTPUT_DIR / 'checkpoint_diagnostics'
diagnostic_df = diagnose_checkpoint_age_sweep(
    checkpoint_path=CHECKPOINT_PATH,
    bundle=bundle,
    source_image=SOURCE_IMAGE,
    source_age=SOURCE_AGE,
    target_ages=[30, 35, 40, 50, 65],
    output_dir=DIAGNOSTIC_DIR,
    use_inverse_diffusion=USE_INVERSE_DIFFUSION,
    num_inference_steps=50,
    strength=0.45,
    text_guidance_scale=7.0,
    image_guidance_scale=1.5,
    seed=2026,
    image_size=256,
)
print(diagnostic_df.to_string(index=False))
print('CSV:', diagnostic_df.attrs['csv_path'])
print('Grid:', diagnostic_df.attrs['grid_path'])
display(diagnostic_df)
display(Image.open(diagnostic_df.attrs['grid_path']))

## 4. Compare direct and inverse

This uses the same source, prompt, seed and guidance settings and saves `source | direct | inverse`. It is the fastest way to judge whether inversion improves identity and structure preservation.

In [ ]:
comparison = compare_inference_modes(
    bundle=bundle,
    image=SOURCE_IMAGE,
    target_age=TARGET_AGE,
    source_age=SOURCE_AGE,
    num_inference_steps=50,
    strength=0.45,
    text_guidance_scale=7.0,
    image_guidance_scale=1.5,
    seed=SEED,
    image_size=256,
    output_path=OUTPUT_DIR / 'source_direct_inverse.png',
)
display(comparison['grid'])

## 5. Optional age sweep

Every output is generated from the same source image. The helper calls the main inference API and preserves result order and age metadata.

In [ ]:
sweep = generate_age_sweep(
    bundle=bundle,
    image=SOURCE_IMAGE,
    ages=[35, 45, 55, 65, 75],
    mode='inverse' if USE_INVERSE_DIFFUSION else 'direct',
    source_age=SOURCE_AGE,
    num_inference_steps=50,
    strength=0.45,
    seed=SEED,
    image_size=256,
    output_path=OUTPUT_DIR / 'age_sweep.png',
)
display(sweep['grid'])

## 7. Numerical preflight

The local preflight checks prompt construction and the analytical CFG formula. Real SD1.5/GPU execution is only marked as passed when explicitly supplied and run on the server.

In [ ]:
run_inference_pipeline_validation()